In [0]:
dbutils.widgets.text("catalog", "fk_ce1366")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

df = spark.read.table(f"{catalog}.bronze.orders")

#casting
df = (
    df.withColumn("order_id", col("order_id").cast("string"))
    .withColumn("customer_id", col("customer_id").cast("string"))
    .withColumn("store_id", col("store_id").cast("string"))
    .withColumn("order_ts", try_to_timestamp(col("order_ts")))
    .withColumn("order_date", to_date(col("order_ts")))
    .withColumn("order_status", col("order_status").cast("string"))
    .withColumn("payment_method", col("payment.method").cast("string"))
    .withColumn("promised_mins", col("delivery.promised_mins").cast("int"))
    .withColumn("actual_mins", col("delivery.actual_mins").cast("int"))
    .withColumn("delivery_fee", col("delivery.delivery_fee").cast("decimal(10,2)"))
    .withColumn("updated_at", try_to_timestamp(col("updated_at")))
)

df = df.drop("ingested_at")
df = df.drop("_source_file")
df = df.drop("tags")
df = df.drop("payment")
df = df.drop("delivery")
df = df.drop("item")
df = df.drop("_rescued_data")


df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(f"{catalog}.silver.orders")

In [0]:
from pyspark.sql.functions import *

df = spark.read.table(f"{catalog}.bronze.orders")

df = df.withColumn("item", explode(col("items"))).drop("items")

df = (
    df.withColumn("order_id", col("order_id").cast("string"))
    .withColumn("product_id", col("item.product_id").cast("string"))
    .withColumn("quantity",
                when(col("item.qty") > 0, col("item.qty"))
                .otherwise(lit(None)))
    .withColumn("unit_price",
                when(col("item.unit_price") > 0, col("item.unit_price"))
                .otherwise(lit(None)))
    .withColumn("discount_amount",
                when(col("item.discount") > 0, col("item.discount"))
                .otherwise(lit(None)))
    .withColumn("gross_amount", (col("quantity") * col("unit_price")))
    .withColumn("net_amount", (col("gross_amount") - col("discount_amount")))

)

df = (
    df.drop("ingested_at")
    .drop("_rescued_data")
    .drop("_source_file")
    .drop("tags")
    .drop("payment")
    .drop("delivery")
    .drop("item")
    .drop("updated_at")
    .drop("order_ts")
    .drop("order_status")
    .drop("payment_method")
    .drop("customer_id")
    .drop("store_id")
)

df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(f"{catalog}.silver.order_items")

In [0]:
from pyspark.sql import Window

def duplicate_order_ids():
    df = spark.read.table(f"{catalog}.bronze.orders")
    window = Window.partitionBy("order_id").orderBy(desc("updated_at"))
    df = df.withColumn("rn", row_number().over(window))
    df = df.filter(col("rn") == 1).drop("rn")
    return df

d = duplicate_order_ids()
d.count()

In [0]:
orders_df = duplicate_order_ids()

customers_df = spark.read.table(f"{catalog}.silver.customers").select(col("customer_id").cast("string").alias("valid_customer_id"))
stores_df = spark.read.table(f"{catalog}.silver.stores").select(col("store_id").cast("string").alias("valid_store_id"))
products_df = spark.read.table(f"{catalog}.silver.products").select(col("product_id").cast("string").alias("valid_product_id"))

#customer_id is null and not exists in silver.customers
invalid_customer_id_df = (
    orders_df.filter(col("customer_id").isNull())
    .select(col("order_id"), lit("MISSING_CUSTOMER").alias("reason"))
    .join(broadcast(customers_df), orders_df.order_id == customers_df.valid_customer_id, "left")
    .filter(col("valid_customer_id").isNull())
    .select(col("order_id"), lit("UNKNOWN_CUSTOMER").alias("reason"))
)


#store_id is null and not exists in silver.stores
invalid_store_id_df = (
    orders_df.filter(col("store_id").isNull())
    .join(broadcast(stores_df), orders_df.order_id == stores_df.valid_store_id, "left")
    .filter(col("valid_store_id").isNull())
    .select(col("order_id"), lit("UNKNOWN_STORE").alias("reason"))
)

#stores invalid order_ts if order_ts is not in format yyyy-MM-dd HH:mm:ss or yyyy-MM-dd'T'HH:mm:ss
invalid_order_ts = (
    orders_df.withColumn("invalid_order_ts",
                         when(try_to_timestamp(col("order_ts")).isNotNull(), lit("valid"))
                        .otherwise(lit("invalid")))
            .filter(col("invalid_order_ts") == "invalid")
            .select(col("order_id"), lit("INVALID_TIMESTAMP").alias("reason"))
            )

#stores if items is null, missing or has no elements
invalid_items_df = (
    orders_df.filter(col("items").isNull() | (size(col("items")) == 0))
    .withColumn("order_id", trim(col("order_id").cast("string")))
    .select(col("order_id"), lit("EMPTY_ORDER").alias("reason"))
)

#product_id is null and not exists in silver.stores
invalid_product_id_df = (
    orders_df.filter(col("product_id").isNull())
    .join(broadcast(products_df), orders_df.order_id == products_df.valid_product_id, "left")
    .filter(col("valid_product_id").isNull())
    .select(col("order_id"), lit("UNKNOWN_PRODUCTS").alias("reason"))
)

# Union all invalid DataFrames and write to silver.quarantine_orders
quarantine_orders_df = (
    invalid_customer_id_df
    .unionByName(invalid_store_id_df)
    .unionByName(invalid_order_ts)
    .unionByName(invalid_items_df)
    .unionByName(invalid_product_id_df)
)

#quarantine_orders_df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_orders")


quarantine_orders_df.display()

